# 11 - E11 Retrieval memory that votes in the logit

E11 tests the current post-E10 hypothesis: DeMemte memory is mechanically
alive, but soft token mixing is too weak to change predictions.  Here the
memory contributes a cache/kNN score directly at the classifier decision:

`logits_final = logits_base + alpha_eff(x) * logits_cache`

Base: `e6_ema_kmeans_restart`.  The primary key is `zq_pool`; `z_pool` and
`fused` are ablations.  The `oracle_cache_*` variant is diagnostic only and
must not be reported as a valid test-time method.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'dememte').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)


repo root: /home/nakato/projects/Dememte


In [2]:
import numpy as np
import pandas as pd
import torch

from dememte.config import e6_config, resolve_data_dir
from dememte.data import build_loaders, seed_everything
from dememte.evaluation import evaluate_dememte_suite, evaluate_dememte_tta_suite, signal_curve_rows
from dememte.io import ensure_dir, load_checkpoint, write_csv, write_json
from dememte.models import make_dememte_e6
from dememte.retrieval import RetrievalConfig, RetrievalLogitAdapter, build_labeled_cache

BASE = 'e6_ema_kmeans_restart'
OUT = ensure_dir(ROOT / 'notebooks' / '11_retrieval_memory' / 'out')
E6_OUT = ROOT / 'notebooks' / '06_e6_zq_alignment' / 'out'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


device: cuda


## Data and checkpoint


In [3]:
cfg = e6_config(BASE)
cfg.data_dir = resolve_data_dir(cfg)
seed_everything(cfg.seed)

tr_loader, va_loader, te_loader, meta = build_loaders(
    data_dir=cfg.data_dir,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    val_ratio=cfg.val_ratio,
    split_seed=cfg.split_seed,
    protocol=cfg.benchmark_protocol,
)
print(meta)

ckpt = E6_OUT / BASE / 'best.pt'

def load_base_model():
    model = make_dememte_e6(cfg, device=device)
    load_checkpoint(model, ckpt, device=device, strict=False)
    model.eval()
    return model


{'protocol': 'historical_trainval_resplit', 'split_seed': 42, 'train_size': 1632, 'val_size': 408, 'test_size': 6149}


## Build source and oracle caches


In [4]:
source_model_for_cache = load_base_model()
source_caches = {
    'zq_pool': build_labeled_cache(source_model_for_cache, tr_loader, device=device, key_space='zq_pool', num_classes=cfg.num_classes),
    'z_pool': build_labeled_cache(source_model_for_cache, tr_loader, device=device, key_space='z_pool', num_classes=cfg.num_classes),
    'fused': build_labeled_cache(source_model_for_cache, tr_loader, device=device, key_space='fused', num_classes=cfg.num_classes),
}
oracle_cache = build_labeled_cache(source_model_for_cache, te_loader, device=device, key_space='zq_pool', num_classes=cfg.num_classes)
{k: v.size for k, v in source_caches.items()}, oracle_cache.size


({'zq_pool': 1632, 'z_pool': 1632, 'fused': 1632}, 6149)

## Variants


In [5]:
variants = [
    ('source_cache_zq_pool_fixed_alpha',
     RetrievalConfig(key_space='zq_pool', alpha_mode='fixed', alpha_max=1.0, cache_source=True, episodic_size=0),
     source_caches['zq_pool']),
    ('source_cache_z_pool_fixed_alpha',
     RetrievalConfig(key_space='z_pool', alpha_mode='fixed', alpha_max=1.0, cache_source=True, episodic_size=0),
     source_caches['z_pool']),
    ('source_cache_fused_fixed_alpha',
     RetrievalConfig(key_space='fused', alpha_mode='fixed', alpha_max=1.0, cache_source=True, episodic_size=0),
     source_caches['fused']),
    ('source_cache_zq_pool_unfamiliarity_alpha',
     RetrievalConfig(key_space='zq_pool', alpha_mode='unfamiliarity', alpha_max=1.0, cache_source=True, episodic_size=0),
     source_caches['zq_pool']),
    ('episodic_cache_zq_pool',
     RetrievalConfig(key_space='zq_pool', alpha_mode='unfamiliarity', alpha_max=1.0, cache_source=False),
     None),
    ('dual_cache_zq_pool',
     RetrievalConfig(key_space='zq_pool', alpha_mode='unfamiliarity', alpha_max=1.0, cache_source=True),
     source_caches['zq_pool']),
    ('oracle_cache_zq_pool_diagnostic_only',
     RetrievalConfig(key_space='zq_pool', alpha_mode='fixed', alpha_max=1.0, cache_source=True, episodic_size=0),
     oracle_cache),
]
[(name, cfg_variant.key_space, cfg_variant.alpha_mode) for name, cfg_variant, _ in variants]


[('source_cache_zq_pool_fixed_alpha', 'zq_pool', 'fixed'),
 ('source_cache_z_pool_fixed_alpha', 'z_pool', 'fixed'),
 ('source_cache_fused_fixed_alpha', 'fused', 'fixed'),
 ('source_cache_zq_pool_unfamiliarity_alpha', 'zq_pool', 'unfamiliarity'),
 ('episodic_cache_zq_pool', 'zq_pool', 'unfamiliarity'),
 ('dual_cache_zq_pool', 'zq_pool', 'unfamiliarity'),
 ('oracle_cache_zq_pool_diagnostic_only', 'zq_pool', 'fixed')]

## Run E11


In [6]:
def write_markdown_table(rows, path):
    if not rows:
        path.write_text('', encoding='utf-8')
        return
    df = pd.DataFrame(rows)
    cols = [
        'variant', 'clean_acc', 'corrupt_acc_avg', 'delta_corrupt_vs_source',
        'ece_corrupt_avg', 'nll_corrupt_avg', 'flip_rate_corrupt_avg',
        'corrected_by_retrieval_corrupt_avg', 'broken_by_retrieval_corrupt_avg',
        'retrieval_alpha_corrupt_avg',
    ]
    cols = [c for c in cols if c in df.columns]
    path.write_text(df[cols].to_markdown(index=False), encoding='utf-8')

all_summaries = []
all_curves = []

src_model = load_base_model()
src_metrics = evaluate_dememte_suite(src_model, te_loader, device=device)
src_clean = src_metrics.pop('clean_record')
src_corrupt = src_metrics.pop('corruption_records')
src_summary = {k: v for k, v in src_metrics.items() if isinstance(v, (int, float, bool, str, np.floating))}
src_summary.update({
    'variant': 'source',
    'label': f'{BASE}::source',
    'base_variant': BASE,
    'base_checkpoint': str(ckpt),
    'protocol': meta['protocol'],
    'split_seed': meta['split_seed'],
    'quantizer_type': cfg.quantizer_type,
    'delta_clean_vs_source': 0.0,
    'delta_corrupt_vs_source': 0.0,
    'diagnostic_only': False,
})
all_summaries.append(src_summary)
all_curves.extend(signal_curve_rows('source', src_summary['label'], src_clean, src_corrupt))

for variant_name, r_cfg, cache in variants:
    print('--', variant_name)

    def factory(r_cfg=r_cfg, cache=cache):
        model = load_base_model()
        return RetrievalLogitAdapter(model, r_cfg, source_cache=cache, num_classes=cfg.num_classes)

    metrics = evaluate_dememte_tta_suite(
        factory,
        te_loader,
        device=device,
        tta_method=variant_name,
        tta_base_variant=BASE,
    )
    clean_record = metrics.pop('clean_record')
    corrupt_records = metrics.pop('corruption_records')
    label = f'{BASE}::{variant_name}'
    curve_rows = signal_curve_rows(variant_name, label, clean_record, corrupt_records)
    summary = {k: v for k, v in metrics.items() if isinstance(v, (int, float, bool, str, np.floating))}
    summary.update({
        'variant': variant_name,
        'label': label,
        'base_variant': BASE,
        'base_checkpoint': str(ckpt),
        'protocol': meta['protocol'],
        'split_seed': meta['split_seed'],
        'quantizer_type': cfg.quantizer_type,
        'delta_clean_vs_source': metrics['clean_acc'] - src_summary['clean_acc'],
        'delta_corrupt_vs_source': metrics['corrupt_acc_avg'] - src_summary['corrupt_acc_avg'],
        'diagnostic_only': variant_name.startswith('oracle_cache'),
    })
    all_summaries.append(summary)
    all_curves.extend(curve_rows)

    method_dir = ensure_dir(OUT / variant_name)
    write_json(summary, method_dir / 'metrics.json')
    write_csv(curve_rows, method_dir / 'signal_curves.csv')

    print({
        k: round(float(summary[k]), 4)
        for k in ['clean_acc', 'corrupt_acc_avg', 'delta_corrupt_vs_source',
                  'flip_rate_corrupt_avg', 'corrected_by_retrieval_corrupt_avg',
                  'broken_by_retrieval_corrupt_avg']
        if k in summary
    })

ranked = sorted(all_summaries, key=lambda r: r.get('corrupt_acc_avg', 0.0), reverse=True)
write_csv(ranked, OUT / 'e11_results.csv')
write_csv(all_curves, OUT / 'e11_curves.csv')
write_markdown_table(ranked, OUT / 'e11_summary.md')
pd.DataFrame(ranked).head(20)


-- source_cache_zq_pool_fixed_alpha
{'clean_acc': 0.7315, 'corrupt_acc_avg': 0.4582, 'delta_corrupt_vs_source': -0.0447, 'flip_rate_corrupt_avg': 0.3151, 'corrected_by_retrieval_corrupt_avg': 0.0459, 'broken_by_retrieval_corrupt_avg': 0.0907}
-- source_cache_z_pool_fixed_alpha
{'clean_acc': 0.7969, 'corrupt_acc_avg': 0.5328, 'delta_corrupt_vs_source': 0.0299, 'flip_rate_corrupt_avg': 0.1201, 'corrected_by_retrieval_corrupt_avg': 0.0455, 'broken_by_retrieval_corrupt_avg': 0.0156}
-- source_cache_fused_fixed_alpha
{'clean_acc': 0.7687, 'corrupt_acc_avg': 0.5127, 'delta_corrupt_vs_source': 0.0098, 'flip_rate_corrupt_avg': 0.1119, 'corrected_by_retrieval_corrupt_avg': 0.0312, 'broken_by_retrieval_corrupt_avg': 0.0214}
-- source_cache_zq_pool_unfamiliarity_alpha
{'clean_acc': 0.7598, 'corrupt_acc_avg': 0.4925, 'delta_corrupt_vs_source': -0.0105, 'flip_rate_corrupt_avg': 0.2043, 'corrected_by_retrieval_corrupt_avg': 0.0336, 'broken_by_retrieval_corrupt_avg': 0.0441}
-- episodic_cache_zq_pool

,clean_acc,corrupt_acc_avg,corrupt_acc_gaussian_noise,corrupt_acc_pixel_mask,corrupt_acc_cutout,corrupt_acc_blur,ece_clean,ece_corrupt_avg,nll_clean,nll_corrupt_avg,...,variant,label,base_variant,base_checkpoint,protocol,split_seed,quantizer_type,delta_clean_vs_source,delta_corrupt_vs_source,diagnostic_only
0,0.796878,0.532824,0.367702,0.368840,0.687591,0.707161,0.089383,0.132385,0.909033,2.008953,...,source_cache_z_pool_fixed_alpha,e6_ema_kmeans_restart::source_cache_z_pool_fix...,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,0.044560,0.029869,False
1,0.813628,0.516588,0.328726,0.299344,0.701090,0.737193,0.072944,0.207803,0.808422,2.283377,...,episodic_cache_zq_pool,e6_ema_kmeans_restart::episodic_cache_zq_pool,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,0.061311,0.013634,False
2,0.768743,0.512726,0.355884,0.355884,0.654686,0.684447,0.124760,0.186793,1.116665,2.233380,...,source_cache_fused_fixed_alpha,e6_ema_kmeans_restart::source_cache_fused_fixe...,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,0.016425,0.009771,False
3,0.809400,0.508416,0.319673,0.290942,0.694801,0.728248,0.095436,0.241479,0.875150,2.411218,...,dual_cache_zq_pool,e6_ema_kmeans_restart::dual_cache_zq_pool,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,0.057082,0.005462,False
4,0.752317,0.502954,0.353445,0.348675,0.638911,0.670787,0.058221,0.090268,0.976969,2.022439,...,source,e6_ema_kmeans_restart::source,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,0.000000,0.000000,False
5,0.787770,0.494660,0.322817,0.293002,0.659890,0.702933,0.136613,0.227294,1.079666,2.477357,...,oracle_cache_zq_pool_diagnostic_only,e6_ema_kmeans_restart::oracle_cache_zq_pool_di...,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,0.035453,-0.008294,True
6,0.759798,0.492465,0.330027,0.325310,0.641568,0.672955,0.103831,0.170954,1.039973,2.186489,...,source_cache_zq_pool_unfamiliarity_alpha,e6_ema_kmeans_restart::source_cache_zq_pool_un...,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,0.007481,-0.010490,False
7,0.731501,0.458205,0.296905,0.288394,0.610018,0.637502,0.163266,0.241852,1.406972,2.626393,...,source_cache_zq_pool_fixed_alpha,e6_ema_kmeans_restart::source_cache_zq_pool_fi...,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,-0.020816,-0.044750,False


## Interpretation guardrails

- `oracle_cache_zq_pool_diagnostic_only` uses test labels and is only a
  headroom diagnostic.
- A positive E11 result must improve or repair flips with bounded ECE/NLL;
  cache activity alone is not enough.
- Before claiming a stable gain, rerun with multiple batch-order seeds.
